# PCB Vision AI ? Complete implementation and academic notebook

This notebook contains the actual dataset parser, patch generator, source-group split, ANN/CNN definitions, training, evaluation, controlled experiments, inference, and localization code. It does not import `src` modules. Generated outputs are intentionally empty until execution. Synthetic arithmetic examples teach mathematics; they are not PCB results.

Run in the project `.venv` with all requirements installed. Download or manually extract DeepPCB, inspect its real files, and confirm the explicit schema/origin mapping before preprocessing. Training and all experiments can take hours on a CPU. See section 6 for controls. GitHub/PyPI were unavailable during generation; this notebook has not been executed end to end.

## 1. Project Introduction

PCB manufacturing defects can break conductors, connect separate nets, or leave unwanted copper. We study visual patch classification with a custom CNN; electrical continuity and industrial reliability require separate validation.


In [ ]:
import sys
from pathlib import Path
print('Interpreter:', sys.executable)
if '.venv' not in Path(sys.executable).parts:
    raise RuntimeError('Select the PCB-Vision-AI .venv interpreter before executing this notebook.')


## 2. Problem Statement

Given a grayscale PCB patch, predict one of seven classes. A full board may contain multiple defects, so its inspection is a separate experimental sliding-window task. Ground-truth boxes are used to form supervised crops, not supplied to the classifier at inference.


## 3. Project Objectives

Train from scratch, avoid group leakage, quantify class-wise behavior, compare ANN/CNN, compare Adam/SGD, and measure augmentation effects. Preserve configurations and measured outputs instead of inventing accuracy.


## 4. Deep Learning Fundamentals

An artificial neuron computes $z=\sum_i w_i x_i+b$ and applies an activation. ReLU is $\max(0,z)$. Softmax is $p_k=\exp(z_k)/\sum_j\exp(z_j)$. The example below is explicit ANN arithmetic on invented teaching inputs; it is not an inspection prediction.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

def neuron(inputs, weights, bias):
    return float(np.dot(inputs, weights) + bias)

def relu(values):
    return np.maximum(0, values)

def stable_softmax(logits):
    shifted = np.asarray(logits) - np.max(logits)
    return np.exp(shifted) / np.exp(shifted).sum()

print('Teaching neuron:', neuron([0.2, 0.8], [0.5, -0.3], 0.1))
print('Teaching softmax:', stable_softmax([1.0, 2.0, 0.5]))


## 5. Dataset Introduction

DeepPCB supplies aligned test/template pairs and six bounding-box defect IDs. Normal is generated outside annotations. The map keeps source IDs separate from display class indices; notably copper and pin-hole appear in a different display order.


## 6. Dataset Download and Setup

The following cells define central settings and the real downloader inline. If downloading is blocked, extract the official ZIP and set `DATASET_DIRECTORY` to its actual `PCBData` directory. `CONFIRM_OBSERVED_SCHEMA` must be set only after section 7 displays real annotation lines and README evidence. Changing epochs changes the experiment budget and is recorded.


In [ ]:
"""Reproducible paths, label mapping, preprocessing, training, and inspection settings."""
from __future__ import annotations
from dataclasses import asdict, dataclass, replace
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
CLASS_NAMES = ('Open Circuit', 'Short Circuit', 'Mouse Bite', 'Spur', 'Pinhole', 'Spurious Copper', 'Normal / Defect-Free')
DEEPPCB_TO_CLASS = {1: 0, 2: 1, 3: 2, 4: 3, 5: 5, 6: 4}

@dataclass(frozen=True)
class Settings:
    root: Path = PROJECT_ROOT
    dataset_dir: Path = PROJECT_ROOT / 'data' / 'raw' / 'DeepPCB-master' / 'PCBData'
    processed_dir: Path = PROJECT_ROOT / 'data' / 'processed'
    models_dir: Path = PROJECT_ROOT / 'models'
    figures_dir: Path = PROJECT_ROOT / 'results' / 'figures'
    metrics_dir: Path = PROJECT_ROOT / 'results' / 'metrics'
    image_size: int = 128
    channels: int = 1
    class_names: tuple[str, ...] = CLASS_NAMES
    batch_size: int = 32
    epochs: int = 40
    learning_rate: float = 0.001
    sgd_learning_rate: float = 0.001
    optimizer: str = 'adam'
    seed: int = 42
    train_fraction: float = 0.7
    validation_fraction: float = 0.15
    test_fraction: float = 0.15
    padding: int = 12
    normal_window_size: int = 128
    normal_patches_per_image: int = 4
    normal_clearance: int = 12
    normal_max_attempts: int = 400
    annotation_origin: str = 'zero'
    group_by_board: bool = True
    augmentation: bool = True
    dropout: float = 0.3
    early_stopping_patience: int = 7
    reduce_lr_patience: int = 3
    confidence_threshold: float = 0.8
    window_size: int = 128
    stride: int = 64
    nms_iou_threshold: float = 0.3
    localization_iou_threshold: float = 0.5
    max_image_pixels: int = 16000000

    def validate(self) -> None:
        if self.image_size != 128 or self.channels != 1 or len(self.class_names) != 7:
            raise ValueError('This architecture requires 128 x 128 x 1 inputs and seven classes.')
        if abs(self.train_fraction + self.validation_fraction + self.test_fraction - 1) > 1e-08:
            raise ValueError('Split fractions must sum to one.')
        if min(self.train_fraction, self.validation_fraction, self.test_fraction) <= 0:
            raise ValueError('All three split fractions must be positive.')
        if self.annotation_origin not in {'zero', 'one'}:
            raise ValueError('annotation_origin must be zero or one.')
        if self.batch_size < 1 or self.epochs < 1 or self.learning_rate <= 0:
            raise ValueError('Batch size, epochs, and learning rate must be positive.')
        if self.optimizer not in {'adam', 'sgd'}:
            raise ValueError('optimizer must be adam or sgd.')
        if min(self.padding, self.normal_clearance, self.normal_patches_per_image) < 0:
            raise ValueError('Padding, clearance, and sample count cannot be negative.')
        if self.normal_window_size < 1 or self.window_size < 1 or (not 1 <= self.stride <= self.window_size):
            raise ValueError('Window sizes must be positive; stride must be within window size.')
        for value in (self.confidence_threshold, self.nms_iou_threshold, self.localization_iou_threshold):
            if not 0 <= value <= 1:
                raise ValueError('Confidence and IoU thresholds must lie within [0, 1].')

    @property
    def model_path(self) -> Path:
        return self.models_dir / 'cnn_adam_aug' / 'best.keras'

    def with_dataset(self, path: str | Path) -> Settings:
        return replace(self, dataset_dir=Path(path).expanduser().resolve())

    def to_dict(self) -> dict:
        values = {key: str(value) if isinstance(value, Path) else value for key, value in asdict(self).items()}
        values['model_output_path'] = str(self.model_path)
        return values
CONFIG = Settings()


In [ ]:
DATASET_DIRECTORY = CONFIG.dataset_dir
CONFIRM_OBSERVED_SCHEMA = False
RUN_TRAINING = True
RUN_FULL_EXPERIMENTS = True
# To shorten a smoke run, use dataclasses.replace(CONFIG, epochs=1) and label it a smoke run.
CONFIG.validate()


In [ ]:
"""Download and safely extract the official DeepPCB source archive (no credentials)."""
from __future__ import annotations
import argparse
import json
import logging
from pathlib import Path
import sys
import urllib.request
import zipfile
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
URL = 'https://codeload.github.com/tangsanli5201/DeepPCB/zip/refs/heads/master'

def download_dataset(destination: Path) -> Path:
    destination = destination.expanduser().resolve()
    destination.mkdir(parents=True, exist_ok=True)
    extracted = destination / 'DeepPCB-master' / 'PCBData'
    if extracted.is_dir() and any(extracted.rglob('*_test.*')):
        logging.info('Existing extracted dataset preserved: %s', extracted)
        return extracted
    archive = destination / 'DeepPCB-source.zip'
    partial = destination / 'DeepPCB-source.zip.part'
    if not archive.is_file():
        try:
            with urllib.request.urlopen(URL, timeout=30) as response, partial.open('wb') as output:
                total = 0
                while (chunk := response.read(1024 * 1024)):
                    total += len(chunk)
                    if total > 5 * 1024 ** 3:
                        raise ValueError('Download exceeded 5 GiB; check the upstream archive.')
                    output.write(chunk)
            partial.replace(archive)
        except Exception:
            partial.unlink(missing_ok=True)
            raise
    with zipfile.ZipFile(archive) as package:
        for member in package.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Unsafe archive path: {member.filename}')
            if member.external_attr >> 16 & 61440 == 40960:
                raise ValueError(f'Archive symlinks are unsupported: {member.filename}')
        package.extractall(destination)
    if not extracted.is_dir():
        raise ValueError('Archive extracted but PCBData is absent. Inspect upstream layout and supply --dataset manually.')
    (destination / 'download_provenance.json').write_text(json.dumps({'source': URL, 'repository': 'https://github.com/tangsanli5201/DeepPCB', 'archive': archive.name}, indent=2), encoding='utf-8')
    return extracted


In [ ]:
if not DATASET_DIRECTORY.is_dir():
    DATASET_DIRECTORY = download_dataset(CONFIG.root / 'data' / 'raw')
CONFIG = CONFIG.with_dataset(DATASET_DIRECTORY)
print('Dataset path:', CONFIG.dataset_dir)


## 7. Dataset Structure Inspection

This full loader discovers actual filenames rather than assuming a single sibling folder layout. Rows must have five integers; unknown IDs and ambiguous files fail. Verify mapping/origin against the official README. Splitting is done now, before the demonstration crops later.


In [ ]:
"""Inspect DeepPCB on disk, validate its text annotations, and split source groups.

Repository access was unavailable during development. This loader recognizes the
published five-integer xyxy/type schema, inspects actual files at runtime, and
fails on a different schema. The label map and coordinate origin remain explicit.
"""
from __future__ import annotations
import argparse
from collections import Counter, defaultdict
from dataclasses import dataclass
import json
import logging
from pathlib import Path
import random
from typing import Iterable
LOGGER = logging.getLogger(__name__)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.pgm'}

class DatasetError(ValueError):
    """A dataset cannot safely be interpreted or split."""

@dataclass(frozen=True)
class Annotation:
    box: tuple[int, int, int, int]
    class_id: int
    source_label: int

@dataclass(frozen=True)
class PCBRecord:
    image_id: str
    group_id: str
    test_path: Path
    template_path: Path
    annotation_path: Path
    annotations: tuple[Annotation, ...]

def validate_box(box: tuple[int, int, int, int], width: int, height: int) -> None:
    """Validate internal zero-based, half-open coordinates without silently clipping."""
    x1, y1, x2, y2 = box
    if width <= 0 or height <= 0 or (not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height)):
        raise DatasetError(f'Invalid bounding box {box} for image {width} x {height}.')

def parse_annotations(path: str | Path, origin: str='zero') -> tuple[Annotation, ...]:
    """Read exactly x_min y_min x_max y_max defect_id integer rows.

    For one-based inclusive input, subtract one from minima only; maxima then
    become exclusive indices. Zero input is interpreted as zero-based xyxy.
    Empty files are valid defect-free annotations; malformed rows are errors.
    """
    path = Path(path)
    if origin not in {'zero', 'one'}:
        raise DatasetError('Coordinate origin must be explicitly zero or one.')
    annotations = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8-sig').splitlines(), 1):
        if not line.strip():
            continue
        tokens = line.split()
        if len(tokens) != 5:
            raise DatasetError(f'{path}:{line_number}: expected five integers, received {line!r}.')
        try:
            x1, y1, x2, y2, source_label = map(int, tokens)
        except ValueError as error:
            raise DatasetError(f'{path}:{line_number}: noninteger annotation {line!r}.') from error
        if source_label not in DEEPPCB_TO_CLASS:
            raise DatasetError(f'{path}:{line_number}: unknown DeepPCB class {source_label}; expected 1..6.')
        if origin == 'one':
            x1, y1 = (x1 - 1, y1 - 1)
        if x1 < 0 or y1 < 0 or x2 <= x1 or (y2 <= y1):
            raise DatasetError(f'{path}:{line_number}: invalid xyxy box {(x1, y1, x2, y2)}.')
        annotations.append(Annotation((x1, y1, x2, y2), DEEPPCB_TO_CLASS[source_label], source_label))
    return tuple(annotations)

def resolve_dataset_root(directory: str | Path) -> Path:
    directory = Path(directory).expanduser().resolve()
    if not directory.is_dir():
        raise DatasetError(f'Dataset directory does not exist: {directory}. See docs/DATASET_DOCUMENTATION.md.')
    if (directory / 'PCBData').is_dir():
        return directory / 'PCBData'
    candidates = sorted(directory.glob('DeepPCB*/PCBData'))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise DatasetError('Multiple extracted datasets found; supply the exact PCBData path.')
    return directory

def pair_identifier(path: Path) -> str:
    if not path.stem.lower().endswith('_test'):
        raise DatasetError(f'Test image does not use the _test suffix: {path}')
    return path.stem[:-5]

def board_group(path: Path, root: Path, by_board: bool=True) -> str:
    relative = path.relative_to(root)
    if by_board:
        for part in relative.parts[:-1]:
            if part.lower().startswith('group'):
                return part
    return str(relative.with_name(pair_identifier(path))).replace('\\', '/')

def load_dataset(directory: str | Path, origin: str='zero', group_by_board: bool=True, strict: bool=False) -> list[PCBRecord]:
    """Discover test/temp pairs and annotation files by actual basenames and proximity.

    Missing pairs or invalid annotations are logged and skipped unless strict.
    Corrupt image decoding and image-bound validation occur during preprocessing.
    Ambiguous annotations are never guessed.
    """
    root = resolve_dataset_root(directory)
    text_index = defaultdict(list)
    for path in root.rglob('*.txt'):
        text_index[path.stem].append(path)
    records = []
    errors = []
    tests = sorted((path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and path.stem.lower().endswith('_test')))
    for test in tests:
        try:
            stem = pair_identifier(test)
            templates = [path for path in test.parent.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and (path.stem == stem + '_temp')]
            if len(templates) != 1:
                raise DatasetError(f'Expected exactly one aligned _temp image for {test}; found {len(templates)}.')
            annotations = text_index.get(stem, []) + text_index.get(stem + '_test', [])
            if len(annotations) > 1:
                local = [path for path in annotations if path.parent.parent == test.parent.parent]
                annotations = local if len(local) == 1 else annotations
            if len(annotations) != 1:
                raise DatasetError(f'Expected one annotation for {test}; found {len(annotations)}.')
            annotation_path = annotations[0]
            image_id = str(test.relative_to(root).with_suffix('')).replace('\\', '/')
            records.append(PCBRecord(image_id, board_group(test, root, group_by_board), test, templates[0], annotation_path, parse_annotations(annotation_path, origin)))
        except (DatasetError, OSError) as error:
            errors.append(str(error))
            LOGGER.warning('Skipping dataset record: %s', error)
            if strict:
                raise DatasetError(str(error)) from error
    if not records:
        detail = ' '.join(errors[:3])
        raise DatasetError(f'No valid _test/_temp annotated pairs found under {root}. {detail}')
    return records

def split_groups(records: Iterable[PCBRecord], settings: Settings=CONFIG) -> dict[str, list[PCBRecord]]:
    """Shuffle indivisible PCB groups before any cropping; reserve nonempty splits."""
    settings.validate()
    records = list(records)
    groups = sorted({record.group_id for record in records})
    if len(groups) < 3:
        raise DatasetError(f'At least three independent groups are required; found {len(groups)}.')
    random.Random(settings.seed).shuffle(groups)
    train_count = min(len(groups) - 2, max(1, round(len(groups) * settings.train_fraction)))
    validation_count = min(len(groups) - train_count - 1, max(1, round(len(groups) * settings.validation_fraction)))
    assignment = {group: 'train' for group in groups[:train_count]}
    assignment.update({group: 'validation' for group in groups[train_count:train_count + validation_count]})
    assignment.update({group: 'test' for group in groups[train_count + validation_count:]})
    splits = {name: [record for record in records if assignment[record.group_id] == name] for name in ('train', 'validation', 'test')}
    assert_no_leakage(splits)
    return splits

def assert_no_leakage(splits: dict[str, list[PCBRecord]]) -> None:
    seen_groups, seen_images, seen_paths = ({}, {}, {})
    for split, records in splits.items():
        for record in records:
            for value, seen, kind in [(record.group_id, seen_groups, 'group'), (record.image_id, seen_images, 'image'), (str(record.test_path.resolve()), seen_paths, 'test path'), (str(record.template_path.resolve()), seen_paths, 'template path')]:
                if value in seen and seen[value] != split:
                    raise DatasetError(f'Leakage: {kind} {value!r} appears in {seen[value]} and {split}.')
                seen[value] = split

def inspect_dataset(directory: str | Path, origin: str='zero', group_by_board: bool=True) -> dict:
    """Return observed paths, raw annotation examples, label counts, and README evidence."""
    root = resolve_dataset_root(directory)
    records = load_dataset(root, origin, group_by_board)
    counts = Counter((annotation.source_label for record in records for annotation in record.annotations))
    readmes = sorted({*root.glob('README*'), *root.parent.glob('README*')})
    summary = {'dataset_root': str(root), 'image_pairs': len(records), 'images': 2 * len(records), 'groups': len({record.group_id for record in records}), 'annotation_schema': 'five integer tokens: xmin ymin xmax ymax type', 'coordinate_origin': origin, 'counts_by_source_label': {str(key): counts.get(key, 0) for key in DEEPPCB_TO_CLASS}, 'counts_by_class': {CLASS_NAMES[value]: counts.get(key, 0) for key, value in DEEPPCB_TO_CLASS.items()}, 'missing_defect_ids': [key for key in DEEPPCB_TO_CLASS if not counts[key]], 'mapping': {str(key): CLASS_NAMES[value] for key, value in DEEPPCB_TO_CLASS.items()}, 'samples': [{'test': str(record.test_path.relative_to(root)), 'template': str(record.template_path.relative_to(root)), 'annotation': str(record.annotation_path.relative_to(root)), 'group': record.group_id, 'raw_annotation_lines': record.annotation_path.read_text().splitlines()[:5]} for record in records[:6]], 'readme_evidence': [{'path': str(path), 'excerpt': path.read_text(encoding='utf-8', errors='replace')[:20000]} for path in readmes if path.is_file()], 'validation_note': 'Observed text schema and labels validated. Image decoding and all bounding boxes are validated by preprocessing. Confirm mapping/origin against upstream README before training.'}
    return summary


In [ ]:
dataset_summary = inspect_dataset(CONFIG.dataset_dir, CONFIG.annotation_origin, CONFIG.group_by_board)
display(dataset_summary)
records = load_dataset(CONFIG.dataset_dir, CONFIG.annotation_origin, CONFIG.group_by_board)
splits = split_groups(records, CONFIG)
assert_no_leakage(splits)
print({name: len(items) for name, items in splits.items()})


## 8. Exploratory Data Analysis

The complete analysis functions below generate observed class counts and annotated images. Class support matters because an overall accuracy can hide a rare class. EDA observes label distribution, never uses test performance to choose training settings.


In [ ]:
"""Generate honest dataset summaries, class distributions, and annotated image plots."""
from __future__ import annotations
import argparse
from collections import Counter
import json
import logging
from pathlib import Path
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
COLORS = ['#15836b', '#ca463a', '#3476aa', '#cf862a', '#835aa3', '#576a72', '#718079']

def plot_annotations(records, output: Path, limit: int=6) -> Path:
    selected = records[:limit]
    if not selected:
        raise ValueError('No records available for annotation visualization.')
    figure, axes = plt.subplots(2, 3, figsize=(14, 9), squeeze=False)
    for axis in axes.flat:
        axis.axis('off')
    for axis, record in zip(axes.flat, selected):
        try:
            image = read_image(record.test_path)
            axis.imshow(image, cmap='gray')
            for annotation in record.annotations:
                x1, y1, x2, y2 = annotation.box
                color = COLORS[annotation.class_id]
                axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.5))
                axis.text(x1, y1, CLASS_NAMES[annotation.class_id], fontsize=7, color='white', backgroundcolor=color)
            axis.set_title(record.image_id, fontsize=9)
        except ValueError as error:
            axis.text(0.5, 0.5, f'Cannot decode sample\n{error}', ha='center', wrap=True)
    figure.tight_layout()
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output, dpi=150)
    plt.close(figure)
    return output

def plot_class_distribution(frame: pd.DataFrame, output: Path) -> Path:
    counts = pd.crosstab(frame['class_id'], frame['split']).reindex(range(7), fill_value=0)
    counts.index = CLASS_NAMES
    figure, axis = plt.subplots(figsize=(12, 5))
    counts.reindex(columns=('train', 'validation', 'test'), fill_value=0).plot.bar(ax=axis, color=['#15836b', '#597b92', '#c48b47'], rot=25)
    axis.set_ylabel('Prepared patches')
    axis.set_xlabel('')
    axis.set_title('Class distribution after splitting source PCB groups')
    figure.tight_layout()
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output, dpi=150)
    plt.close(figure)
    return output

def analyze_dataset(settings=CONFIG) -> dict:
    inspection = inspect_dataset(settings.dataset_dir, settings.annotation_origin, settings.group_by_board)
    records = load_dataset(settings.dataset_dir, settings.annotation_origin, settings.group_by_board)
    settings.metrics_dir.mkdir(parents=True, exist_ok=True)
    (settings.metrics_dir / 'dataset_summary.json').write_text(json.dumps(inspection, indent=2), encoding='utf-8')
    plot_annotations(records, settings.figures_dir / 'sample_annotations.png')
    figure, axis = plt.subplots(figsize=(10, 5))
    names, counts = zip(*inspection['counts_by_class'].items())
    axis.bar(names, counts, color='#15836b')
    axis.set_ylabel('Annotated defects')
    axis.tick_params(axis='x', rotation=25)
    axis.set_title('Observed DeepPCB annotation distribution')
    figure.tight_layout()
    figure.savefig(settings.figures_dir / 'annotation_distribution.png', dpi=150)
    plt.close(figure)
    if (settings.processed_dir / 'manifest.csv').is_file():
        frame, metadata = load_manifest(settings)
        plot_class_distribution(frame, settings.figures_dir / 'class_distribution.png')
        inspection['prepared_counts'] = metadata['counts_by_split']
    return inspection


In [ ]:
observed_counts = pd.Series(dataset_summary['counts_by_class'], name='Annotated defects')
display(observed_counts)
observed_counts.plot.barh(color='#15836b', title='Actual observed defect counts')
plt.show()


## 9. Image Visualization

Open actual images and draw their annotated rectangles. The helper decodes Unicode Windows paths, verifies the image, and rejects oversized/corrupt input. No fabricated PCB pictures are used.


In [ ]:
import cv2
def read_image(path: str | Path, max_pixels: int=CONFIG.max_image_pixels) -> np.ndarray:
    """Decode grayscale with Unicode-safe paths and reject corrupt/oversized images."""
    path = Path(path)
    try:
        with Image.open(path) as image:
            if image.width * image.height > max_pixels:
                raise ValueError(f'Image exceeds the {max_pixels:,}-pixel limit: {path}')
            image.verify()
        image = cv2.imdecode(np.frombuffer(path.read_bytes(), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    except (OSError, ValueError) as error:
        raise ValueError(f'Cannot read image {path}: {error}') from error
    if image is None or image.size == 0:
        raise ValueError(f'OpenCV could not decode {path}.')
    return image
plot_annotations(records, CONFIG.figures_dir / 'sample_annotations.png')
display(Image.open(CONFIG.figures_dir / 'sample_annotations.png'))


## 10. Annotation Parsing

The implementation is in section 7: `parse_annotations` reads exact integer rows and maps IDs; `validate_box` enforces positive half-open bounds within image dimensions. One-based inclusive coordinates are converted explicitly when configured. Display and validate a real sample below.


In [ ]:
sample_record = next(record for record in splits['train'] if record.annotations)
print(sample_record.annotation_path.read_text())
sample_image = read_image(sample_record.test_path)
for annotation in sample_record.annotations:
    validate_box(annotation.box, sample_image.shape[1], sample_image.shape[0])
    print(annotation)


## 11. Image Preprocessing

The complete pipeline below converts grayscale/RGB input consistently, resizes, and normalizes. For 8-bit pixels, $x_{norm}=x/255$. Already normalized floating data is not divided twice. Prepared PNGs preserve uint8 intensities; TensorFlow divides once when decoding.


In [ ]:
"""Leakage-safe patch preparation and one preprocessing contract for training/inference."""
from __future__ import annotations
import argparse
from collections import Counter
import csv
from dataclasses import replace
import hashlib
import json
import logging
from pathlib import Path
import uuid
import cv2
import numpy as np
from PIL import Image
LOGGER = logging.getLogger(__name__)

def read_image(path: str | Path, max_pixels: int=CONFIG.max_image_pixels) -> np.ndarray:
    """Decode grayscale with Unicode-safe paths and reject corrupt/oversized images."""
    path = Path(path)
    try:
        with Image.open(path) as image:
            if image.width * image.height > max_pixels:
                raise ValueError(f'Image exceeds the {max_pixels:,}-pixel limit: {path}')
            image.verify()
        image = cv2.imdecode(np.frombuffer(path.read_bytes(), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    except (OSError, ValueError) as error:
        raise ValueError(f'Cannot read image {path}: {error}') from error
    if image is None or image.size == 0:
        raise ValueError(f'OpenCV could not decode {path}.')
    return image

def preprocess_image(image: np.ndarray, image_size: int=CONFIG.image_size) -> np.ndarray:
    """Return finite float32 [0,1] grayscale HWC data; RGB arrays use RGB ordering."""
    image = np.asarray(image)
    was_integer = np.issubdtype(image.dtype, np.integer)
    if image.size == 0 or image.ndim not in (2, 3) or (not np.issubdtype(image.dtype, np.number)):
        raise ValueError('Expected a nonempty numeric grayscale or RGB image array.')
    if image.ndim == 3:
        if image.shape[2] == 1:
            image = image[..., 0]
        elif image.shape[2] in (3, 4):
            image = cv2.cvtColor(image.astype(np.float32), cv2.COLOR_RGB2GRAY if image.shape[2] == 3 else cv2.COLOR_RGBA2GRAY)
        else:
            raise ValueError('An image must have one, three, or four channels.')
    image = image.astype(np.float32)
    if not np.isfinite(image).all() or image.min() < 0 or image.max() > 255:
        raise ValueError('Pixels must be finite and in [0,255] or already normalized to [0,1].')
    if image.max() > 1 or was_integer:
        image = image / 255.0
    resized = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_AREA if min(image.shape) > image_size else cv2.INTER_LINEAR)
    return np.clip(resized, 0, 1).astype(np.float32)[..., np.newaxis]

def intersection_area(first: tuple[int, int, int, int], second: tuple[int, int, int, int]) -> int:
    return max(0, min(first[2], second[2]) - max(first[0], second[0])) * max(0, min(first[3], second[3]) - max(first[1], second[1]))

def padded_box(box: tuple[int, int, int, int], width: int, height: int, padding: int) -> tuple[int, int, int, int]:
    validate_box(box, width, height)
    if padding < 0:
        raise ValueError('Padding cannot be negative.')
    return (max(0, box[0] - padding), max(0, box[1] - padding), min(width, box[2] + padding), min(height, box[3] + padding))

def extract_defect_patch(image: np.ndarray, box: tuple[int, int, int, int], padding: int=CONFIG.padding, image_size: int=CONFIG.image_size) -> np.ndarray:
    height, width = image.shape[:2]
    x1, y1, x2, y2 = padded_box(box, width, height, padding)
    return preprocess_image(image[y1:y2, x1:x2], image_size)

def generate_normal_boxes(image_shape: tuple[int, ...], boxes: list[tuple[int, int, int, int]], rng: np.random.Generator, window_size: int=CONFIG.normal_window_size, count: int=CONFIG.normal_patches_per_image, clearance: int=CONFIG.normal_clearance, max_attempts: int=CONFIG.normal_max_attempts) -> list[tuple[int, int, int, int]]:
    """Select distinct windows outside all expanded defect boxes; never duplicate to fill quotas."""
    height, width = image_shape[:2]
    if min(window_size, max_attempts) < 1 or min(count, clearance) < 0:
        raise ValueError('Normal sampling settings are invalid.')
    if window_size > min(height, width):
        return []
    forbidden = [padded_box(box, width, height, clearance) for box in boxes]
    samples = []
    for _ in range(max_attempts):
        if len(samples) >= count:
            break
        x = int(rng.integers(0, width - window_size + 1))
        y = int(rng.integers(0, height - window_size + 1))
        candidate = (x, y, x + window_size, y + window_size)
        if candidate in samples or any((intersection_area(candidate, box) for box in forbidden)):
            continue
        area = window_size ** 2
        if any((intersection_area(candidate, box) / area > 0.5 for box in samples)):
            continue
        samples.append(candidate)
    return samples

def _save_gray(path: Path, gray: np.ndarray, image_size: int) -> None:
    image = cv2.resize(gray, (image_size, image_size), interpolation=cv2.INTER_AREA if min(gray.shape) > image_size else cv2.INTER_LINEAR)
    success, encoded = cv2.imencode('.png', image)
    if not success:
        raise ValueError(f'Failed to encode {path}.')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(encoded.tobytes())

def prepare_dataset(settings: Settings=CONFIG, overwrite: bool=False, confirmed_schema: bool=False) -> Path:
    """Inspect, split groups, validate both images, crop, and atomically publish a manifest.

    Confirmation is required because upstream files could not be inspected in the
    restricted development environment. It records the selected label/origin contract.
    Rebuilds use new versioned directories; existing data is never recursively deleted.
    """
    settings.validate()
    destination = settings.processed_dir
    manifest_path = destination / 'manifest.csv'
    if manifest_path.is_file() and (not overwrite):
        load_manifest(settings)
        LOGGER.info('Existing validated manifest preserved: %s', manifest_path)
        return manifest_path
    if not confirmed_schema:
        raise DatasetError('Run src.dataset_loader, inspect its raw lines/README/class map, then pass --confirm-schema after verifying them.')
    inspection = inspect_dataset(settings.dataset_dir, settings.annotation_origin, settings.group_by_board)
    if inspection['missing_defect_ids']:
        raise DatasetError(f"Dataset is missing defect IDs {inspection['missing_defect_ids']}; provide the full dataset.")
    records = load_dataset(settings.dataset_dir, settings.annotation_origin, settings.group_by_board)
    splits = split_groups(records, settings)
    destination.mkdir(parents=True, exist_ok=True)
    build = destination / f'dataset_{settings.seed}_{uuid.uuid4().hex[:8]}'
    build.mkdir()
    rows, skipped = ([], [])
    digest = hashlib.sha256()
    for split_name, split_records in splits.items():
        for record in split_records:
            try:
                test, template = (read_image(record.test_path, settings.max_image_pixels), read_image(record.template_path, settings.max_image_pixels))
                if test.shape != template.shape:
                    raise DatasetError(f'Aligned pair dimensions differ: {record.image_id}.')
                height, width = test.shape
                for annotation in record.annotations:
                    validate_box(annotation.box, width, height)
                for path in (record.test_path, record.template_path, record.annotation_path):
                    digest.update(path.name.encode())
                    digest.update(path.read_bytes())
                seed_material = hashlib.sha256(f'{settings.seed}:{record.image_id}'.encode()).digest()
                rng = np.random.default_rng(int.from_bytes(seed_material[:8], 'little'))
                sample_id = hashlib.sha256(record.image_id.encode()).hexdigest()[:16]
                crops = []
                for index, annotation in enumerate(record.annotations):
                    crop = padded_box(annotation.box, width, height, settings.padding)
                    if any((other.class_id != annotation.class_id and intersection_area(crop, other.box) > 0 for other in record.annotations)):
                        LOGGER.warning('Skipping ambiguous multi-class crop %s annotation %s', record.image_id, index)
                        continue
                    crops.append((f'defect_{index}', annotation.class_id, crop, False))
                normals = generate_normal_boxes(test.shape, [annotation.box for annotation in record.annotations], rng, settings.normal_window_size, settings.normal_patches_per_image, settings.normal_clearance, settings.normal_max_attempts)
                if len(normals) < settings.normal_patches_per_image:
                    LOGGER.warning('Only %s/%s normal windows available in %s; no samples duplicated.', len(normals), settings.normal_patches_per_image, record.image_id)
                crops.extend(((f'normal_{index}', 6, box, True) for index, box in enumerate(normals)))
                for crop_name, class_id, box, is_normal in crops:
                    x1, y1, x2, y2 = box
                    patch_path = build / split_name / str(class_id) / f'{sample_id}_{crop_name}.png'
                    _save_gray(patch_path, test[y1:y2, x1:x2], settings.image_size)
                    rows.append({'patch_path': patch_path.relative_to(destination).as_posix(), 'split': split_name, 'class_id': class_id, 'class_name': CLASS_NAMES[class_id], 'image_id': record.image_id, 'group_id': record.group_id, 'source_image': str(record.test_path.resolve()), 'template_image': str(record.template_path.resolve()), 'annotation_path': str(record.annotation_path.resolve()), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'is_normal': is_normal})
            except (OSError, ValueError, DatasetError) as error:
                skipped.append({'image_id': record.image_id, 'reason': str(error)})
                LOGGER.warning('Skipping invalid pair %s: %s', record.image_id, error)
    if not rows:
        raise DatasetError('No patches extracted. Review the input images, origin, and annotations.')
    counts = {name: Counter((int(row['class_id']) for row in rows if row['split'] == name)) for name in splits}
    if (missing := [CLASS_NAMES[index] for index in range(7) if not counts['train'][index]]):
        raise DatasetError(f'Training split is missing classes {missing}. Use more source groups or another documented seed; do not duplicate validation/test samples.')
    for name in ('validation', 'test'):
        if not sum(counts[name].values()):
            raise DatasetError(f'No usable patches in {name} split.')
        if (missing := [CLASS_NAMES[index] for index in range(7) if not counts[name][index]]):
            LOGGER.warning('%s split has no examples of %s; reports will show zero support.', name, missing)
    temporary = destination / 'manifest.csv.tmp'
    with temporary.open('w', newline='', encoding='utf-8') as output:
        writer = csv.DictWriter(output, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    metadata = {'settings': settings.to_dict(), 'schema_confirmed': True, 'inspection': inspection, 'dataset_fingerprint': digest.hexdigest(), 'manifest_sha256': hashlib.sha256(temporary.read_bytes()).hexdigest(), 'groups_by_split': {name: sorted({record.group_id for record in records}) for name, records in splits.items()}, 'counts_by_split': {name: {CLASS_NAMES[index]: counts[name][index] for index in range(7)} for name in splits}, 'patch_count': len(rows), 'skipped_pairs': skipped, 'build_directory': build.name, 'normal_definition': 'test-image window outside every annotated box plus clearance; assumes annotation completeness'}
    (destination / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    temporary.replace(manifest_path)
    load_manifest(settings)
    LOGGER.info('Prepared %s patches; manifest: %s', len(rows), manifest_path)
    return manifest_path

def load_manifest(settings: Settings=CONFIG):
    """Read a prepared manifest, verify its hash, paths, label range, and split isolation."""
    import pandas as pd
    path = settings.processed_dir / 'manifest.csv'
    metadata_path = settings.processed_dir / 'metadata.json'
    if not path.is_file() or not metadata_path.is_file():
        raise DatasetError('Prepared data is missing. Run python -m src.preprocessing first.')
    metadata = json.loads(metadata_path.read_text())
    if hashlib.sha256(path.read_bytes()).hexdigest() != metadata['manifest_sha256']:
        raise DatasetError('Manifest checksum differs from preparation metadata; rerun preprocessing.')
    if metadata['settings']['image_size'] != settings.image_size or tuple(metadata['settings']['class_names']) != settings.class_names:
        raise DatasetError('Prepared dimensions/class mapping differ from current configuration.')
    frame = pd.read_csv(path)
    required = {'patch_path', 'split', 'class_id', 'image_id', 'group_id', 'source_image', 'template_image'}
    if not required.issubset(frame.columns) or frame.empty:
        raise DatasetError('Manifest columns are missing or manifest is empty.')
    if not frame['class_id'].isin(range(7)).all() or not frame['split'].isin(('train', 'validation', 'test')).all():
        raise DatasetError('Manifest has invalid labels or split names.')
    for column in ('group_id', 'image_id', 'source_image', 'template_image'):
        if (frame.groupby(column)['split'].nunique() > 1).any():
            raise DatasetError(f'Leakage detected for manifest column {column}.')
    if frame['patch_path'].duplicated().any():
        raise DatasetError('Duplicate patch paths detected.')
    for relative in frame['patch_path']:
        resolved = (settings.processed_dir / relative).resolve()
        if not resolved.is_relative_to(settings.processed_dir.resolve()) or not resolved.is_file():
            raise DatasetError(f'Missing or unsafe prepared patch path: {relative}')
    return (frame, metadata)

def make_tf_dataset(frame, settings: Settings=CONFIG, training: bool=False):
    """Stream prepared PNGs; only training is shuffled. Model augmentation is training-only."""
    import tensorflow as tf
    paths = [str((settings.processed_dir / path).resolve()) for path in frame['patch_path']]
    labels = frame['class_id'].to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    def decode(path, label):
        image = tf.io.decode_png(tf.io.read_file(path), channels=1)
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.ensure_shape(image, (settings.image_size, settings.image_size, 1))
        return (image, label)
    options = tf.data.Options()
    options.experimental_deterministic = True
    dataset = dataset.with_options(options).map(decode, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if training:
        dataset = dataset.shuffle(len(frame), seed=settings.seed, reshuffle_each_iteration=True)
    return dataset.batch(settings.batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
normalized = preprocess_image(sample_image)
print(normalized.shape, normalized.dtype, normalized.min(), normalized.max())
assert normalized.shape == (128, 128, 1)
assert np.isfinite(normalized).all()


## 12. Defect Patch Extraction

Crops include configurable clipped padding. A crop containing a different annotated defect class is skipped rather than assigned an ambiguous single label. This example is a training-group crop; the source split was already assigned in section 7.


In [ ]:
example_patch = extract_defect_patch(sample_image, sample_record.annotations[0].box, CONFIG.padding)
plt.imshow(example_patch[..., 0], cmap='gray'); plt.title(CLASS_NAMES[sample_record.annotations[0].class_id]); plt.axis('off'); plt.show()


## 13. Normal Patch Generation

Normal windows come from the test image outside every defect box plus clearance. They are distinct, avoid excessive overlap, and are not duplicated when a requested quota is impossible. This label assumes annotations are complete; it is not an electrical certification.


In [ ]:
normal_boxes = generate_normal_boxes(sample_image.shape, [a.box for a in sample_record.annotations], np.random.default_rng(CONFIG.seed), CONFIG.normal_window_size, CONFIG.normal_patches_per_image, CONFIG.normal_clearance, CONFIG.normal_max_attempts)
print('Actual sampled normal boxes:', normal_boxes)
if normal_boxes:
    x1, y1, x2, y2 = normal_boxes[0]
    plt.imshow(sample_image[y1:y2, x1:x2], cmap='gray'); plt.title('Annotation-derived normal example'); plt.axis('off'); plt.show()


## 14. Dataset Splitting

Grouping defaults to the `group...` board directory, which is stricter than keeping one test/template pair together. Sorted groups are shuffled with a fixed seed and assigned before patch extraction. Validation/test are never augmented or duplicated. Confirm the observed contract, then publish the prepared manifest.


In [ ]:
assert_no_leakage(splits)
if not CONFIRM_OBSERVED_SCHEMA:
    raise RuntimeError('Review section 7 real files and upstream label/origin evidence; set CONFIRM_OBSERVED_SCHEMA=True only after confirming them.')
manifest_path = prepare_dataset(CONFIG, confirmed_schema=True)
frame, preparation = load_manifest(CONFIG)
display(pd.crosstab(frame['class_name'], frame['split']))
plot_class_distribution(frame, CONFIG.figures_dir / 'class_distribution.png')
display(Image.open(CONFIG.figures_dir / 'class_distribution.png'))


## 15. Data Augmentation

The model contains training-only random flips, small rotations/translations/zoom, and bounded brightness variation. Keras bypasses these layers during inference and validation. The exact implementation is shown with the models next; large deformations that can change defect identity are excluded.


## 16. ANN Architecture Implementation

The following inline module defines both architectures and registered sparse multiclass precision/recall metrics. ANN uses flattened 16,384-pixel inputs, Dense256, Dense128, dropout, and seven-class softmax. Its parameter count is comparable to the CNN, so locality and weight sharing are meaningful differences.


In [ ]:
"""Custom CNN/ANN from scratch, sparse multiclass metrics, and architecture diagrams."""
from __future__ import annotations
from dataclasses import replace
from pathlib import Path
import tensorflow as tf

@tf.keras.utils.register_keras_serializable(package='PCBVision')
class SparseMacroPrecision(tf.keras.metrics.Metric):
    """Macro precision based on argmax classes, rather than seven independent binary labels."""

    def __init__(self, num_classes: int=7, name: str='precision', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.confusion = self.add_weight(name='confusion', shape=(num_classes, num_classes), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        labels = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predicted = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = None if sample_weight is None else tf.cast(tf.reshape(sample_weight, [-1]), self.dtype)
        self.confusion.assign_add(tf.math.confusion_matrix(labels, predicted, num_classes=self.num_classes, weights=weights, dtype=self.dtype))

    def result(self):
        return tf.reduce_mean(tf.math.divide_no_nan(tf.linalg.diag_part(self.confusion), tf.reduce_sum(self.confusion, axis=0)))

    def reset_state(self):
        self.confusion.assign(tf.zeros_like(self.confusion))

    def get_config(self):
        return {**super().get_config(), 'num_classes': self.num_classes}

@tf.keras.utils.register_keras_serializable(package='PCBVision')
class SparseMacroRecall(SparseMacroPrecision):

    def __init__(self, num_classes: int=7, name: str='recall', **kwargs):
        super().__init__(num_classes=num_classes, name=name, **kwargs)

    def result(self):
        return tf.reduce_mean(tf.math.divide_no_nan(tf.linalg.diag_part(self.confusion), tf.reduce_sum(self.confusion, axis=1)))

def set_reproducibility(seed: int=CONFIG.seed) -> None:
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()

def augmentation_layers(seed: int=CONFIG.seed):
    """Mild geometry/brightness changes, active only when Keras training=True."""
    return tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=seed, name='flip'), tf.keras.layers.RandomRotation(0.035, fill_mode='reflect', seed=seed + 1, name='rotation'), tf.keras.layers.RandomTranslation(0.04, 0.04, fill_mode='reflect', seed=seed + 2, name='translation'), tf.keras.layers.RandomZoom((-0.05, 0.05), (-0.05, 0.05), fill_mode='reflect', seed=seed + 3, name='zoom'), tf.keras.layers.RandomBrightness(0.08, value_range=(0, 1), seed=seed + 4, name='brightness')], name='training_augmentation')

def build_cnn(settings: Settings=CONFIG, augmentation: bool | None=None) -> tf.keras.Model:
    """128x128x1 -> (Conv/ReLU/Pool)x3 -> Flatten -> Dense128 -> Dropout -> Softmax7."""
    settings.validate()
    set_reproducibility(settings.seed)
    inputs = tf.keras.Input((settings.image_size, settings.image_size, settings.channels), name='pcb_patch')
    use_augmentation = settings.augmentation if augmentation is None else augmentation
    features = augmentation_layers(settings.seed)(inputs) if use_augmentation else inputs
    for block, filters in enumerate((32, 64, 128), 1):
        features = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 10 + block), name=f'conv_{block}')(features)
        features = tf.keras.layers.MaxPooling2D(2, name=f'pool_{block}')(features)
    features = tf.keras.layers.Flatten(name='flatten')(features)
    features = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 20), name='dense_features')(features)
    features = tf.keras.layers.Dropout(settings.dropout, seed=settings.seed + 21, name='dropout')(features)
    probabilities = tf.keras.layers.Dense(7, activation='softmax', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 22), name='defect_probabilities')(features)
    return tf.keras.Model(inputs, probabilities, name='pcb_custom_cnn')

def build_ann(settings: Settings=CONFIG, augmentation: bool=False) -> tf.keras.Model:
    """A parameter-comparable flattened-image baseline with Dense256/Dense128 layers."""
    settings.validate()
    set_reproducibility(settings.seed)
    inputs = tf.keras.Input((128, 128, 1), name='pcb_patch')
    features = augmentation_layers(settings.seed)(inputs) if augmentation else inputs
    features = tf.keras.layers.Flatten(name='flatten')(features)
    features = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 11), name='ann_dense_1')(features)
    features = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 20), name='ann_dense_2')(features)
    features = tf.keras.layers.Dropout(settings.dropout, seed=settings.seed + 21, name='dropout')(features)
    outputs = tf.keras.layers.Dense(7, activation='softmax', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=settings.seed + 22), name='defect_probabilities')(features)
    return tf.keras.Model(inputs, outputs, name='pcb_ann_baseline')

def compile_model(model: tf.keras.Model, settings: Settings=CONFIG) -> tf.keras.Model:
    if settings.optimizer == 'adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=settings.learning_rate)
    elif settings.optimizer == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=settings.learning_rate, momentum=0.0)
    else:
        raise ValueError('Optimizer must be adam or sgd.')
    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'), SparseMacroPrecision(), SparseMacroRecall()])
    return model

def load_keras_model(path: str | Path, compile: bool=False) -> tf.keras.Model:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Trained model not found: {path}. Run python -m src.train first.')
    return tf.keras.models.load_model(path, compile=compile, custom_objects={'SparseMacroPrecision': SparseMacroPrecision, 'SparseMacroRecall': SparseMacroRecall})

def architecture_details(model: tf.keras.Model) -> list[dict]:
    return [{'layer': layer.name, 'type': type(layer).__name__, 'output_shape': [None if dim is None else int(dim) for dim in layer.output.shape], 'parameters': layer.count_params()} for layer in model.layers]

def plot_architecture(model: tf.keras.Model, path: Path) -> Path:
    """Render the actual model graph as a readable layer stack without Graphviz."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    layers = architecture_details(model)
    figure, axis = plt.subplots(figsize=(9, max(6, len(layers) * 0.65)))
    axis.axis('off')
    for index, layer in enumerate(layers):
        y = len(layers) - index
        text = f"{layer['layer']} | {layer['type']}\n{layer['output_shape']} | {layer['parameters']:,} parameters"
        axis.text(0.5, y, text, ha='center', va='center', fontsize=9, bbox={'boxstyle': 'round,pad=0.35', 'facecolor': '#eff4f1', 'edgecolor': '#b7c8c0'})
        if index:
            axis.annotate('', xy=(0.5, y + 0.35), xytext=(0.5, y + 0.65), arrowprops={'arrowstyle': '->', 'color': '#597369'})
    axis.set_xlim(0, 1)
    axis.set_ylim(0.3, len(layers) + 0.7)
    figure.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


In [ ]:
ann_model = build_ann(CONFIG, augmentation=False)
ann_model.summary()


## 17. CNN Architecture Implementation

CNN blocks use 32, 64, 128 filters with 3?3 kernels, ReLU, same padding, and 2?2 max pooling. Flatten gives 32,768 features; Dense128, dropout0.3, and Softmax7 complete the requested head. All initializers have reproducible explicit seeds.


In [ ]:
cnn_model = build_cnn(CONFIG)
cnn_model.summary()
plot_architecture(cnn_model, CONFIG.figures_dir / 'notebook_architecture.png')
display(Image.open(CONFIG.figures_dir / 'notebook_architecture.png'))


## 18. Model Summary

Record each actual Keras layer output and parameter count. A convolution has `(kernel_height * kernel_width * input_channels + 1) * filters` parameters, including bias. A dense layer has `(inputs + 1) * outputs`.


In [ ]:
display(pd.DataFrame(architecture_details(cnn_model)))
print('Trainable parameters:', sum(int(np.prod(v.shape)) for v in cnn_model.trainable_weights))


## 19. Forward Propagation Explanation

Forward propagation applies learned convolutions, ReLU, pooling, dense activations, and softmax in sequence. TensorFlow computes the convolution efficiently; the teaching loop below explicitly computes one valid cross-correlation feature map.


In [ ]:
def teaching_convolution(image, kernel):
    height, width = image.shape
    kh, kw = kernel.shape
    output = np.zeros((height-kh+1, width-kw+1), dtype=float)
    for row in range(output.shape[0]):
        for column in range(output.shape[1]):
            output[row, column] = np.sum(image[row:row+kh, column:column+kw] * kernel)
    return output

print(teaching_convolution(np.arange(25).reshape(5,5), np.array([[1,0,-1],[1,0,-1],[1,0,-1]])))


## 20. Loss Function Explanation

For integer class $y$, sparse categorical cross-entropy is $L=-\log(p_y)$. It penalizes confident wrong predictions more strongly than uncertain ones. All seven probabilities compete through softmax. Class weights apply to training loss only.


In [ ]:
batch_images, batch_labels = next(iter(make_tf_dataset(frame[frame['split']=='train'], CONFIG)))
probabilities = cnn_model(batch_images, training=False)
loss_function = tf.keras.losses.SparseCategoricalCrossentropy()
print('Actual untrained-batch loss (not a project metric):', float(loss_function(batch_labels, probabilities)))


## 21. Backpropagation Explanation

The chain rule computes $\partial L/\partial w$ through every layer. `GradientTape` records the forward operations and returns gradients. The example displays gradient shapes on one real training batch; the controlled training runs create freshly seeded models afterwards.


In [ ]:
with tf.GradientTape() as tape:
    batch_probabilities = cnn_model(batch_images, training=True)
    batch_loss = loss_function(batch_labels, batch_probabilities)
gradients = tape.gradient(batch_loss, cnn_model.trainable_variables)
print([(variable.name, tuple(gradient.shape)) for variable, gradient in zip(cnn_model.trainable_variables, gradients) if gradient is not None])


## 22. Adam Optimizer Implementation

Adam maintains first/second gradient moments: $m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$, $v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$. Bias-correct them before $w\leftarrow w-\eta\hat m/(\sqrt{\hat v}+\epsilon)$. The teaching update below exposes the equations; actual training uses Keras Adam.


In [ ]:
def teaching_adam_step(weight, gradient, first_moment, second_moment, step, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7):
    first_moment = beta1*first_moment + (1-beta1)*gradient
    second_moment = beta2*second_moment + (1-beta2)*gradient**2
    corrected_first = first_moment/(1-beta1**step)
    corrected_second = second_moment/(1-beta2**step)
    updated = weight-learning_rate*corrected_first/(np.sqrt(corrected_second)+epsilon)
    return updated, first_moment, second_moment

print('Teaching Adam:', teaching_adam_step(1.0, 0.2, 0.0, 0.0, 1))
compile_model(cnn_model, CONFIG)
print(cnn_model.optimizer)


## 23. SGD Optimizer Implementation

SGD applies $w\leftarrow w-\eta g$ per mini-batch. This project uses SGD without momentum for a direct introductory comparison and the same initial learning rate as Adam. A fixed budget comparison is not a claim that SGD can never outperform Adam after tuning.


In [ ]:
def teaching_sgd_step(weight, gradient, learning_rate=0.001):
    return weight-learning_rate*gradient

print('Teaching SGD:', teaching_sgd_step(1.0, 0.2))
from dataclasses import replace
sgd_model = compile_model(build_cnn(CONFIG, augmentation=False), replace(CONFIG, optimizer='sgd'))
print(sgd_model.optimizer)


## 24. Model Training

The full training implementation below fits only training patches, calculates training-only class weights, and uses validation-loss early stopping/checkpoint/LR reduction. Test data never controls weights, callbacks, or checkpoint selection. Completed artifacts have checksums and provenance.


In [ ]:
"""Reproducible training with class weights, validation-only callbacks, and provenance."""
from __future__ import annotations
import argparse
from collections import Counter
from dataclasses import replace
import hashlib
import json
import logging
from pathlib import Path
import platform
import re
import time
import numpy as np
import tensorflow as tf
LOGGER = logging.getLogger(__name__)

def class_weights(labels) -> dict[int, float]:
    counts = Counter((int(label) for label in labels))
    if set(counts) != set(range(7)):
        raise ValueError('Training requires all seven classes; class weights cannot repair missing classes.')
    total = sum(counts.values())
    return {class_id: total / (7 * count) for class_id, count in counts.items()}

def train_model(settings: Settings=CONFIG, architecture: str='cnn', run_name: str | None=None, overwrite: bool=False, evaluate: bool=True) -> dict:
    settings.validate()
    if architecture not in {'cnn', 'ann'}:
        raise ValueError('Architecture must be cnn or ann.')
    run_name = run_name or f"{architecture}_{settings.optimizer}_{('aug' if settings.augmentation else 'plain')}"
    if not re.fullmatch('[A-Za-z0-9_-]+', run_name):
        raise ValueError('Run name must contain only letters, numbers, hyphens, or underscores.')
    model_dir, metrics_dir = (settings.models_dir / run_name, settings.metrics_dir / run_name)
    model_path = model_dir / 'best.keras'
    if model_path.exists() and (not overwrite):
        raise FileExistsError(f'Model already exists: {model_path}. Use a new run name or --overwrite explicitly.')
    frame, preparation = load_manifest(settings)
    train_frame = frame[frame['split'] == 'train']
    validation_frame = frame[frame['split'] == 'validation']
    if validation_frame.empty:
        raise ValueError('Validation split is empty.')
    set_reproducibility(settings.seed)
    tf.keras.backend.clear_session()
    model = build_cnn(settings) if architecture == 'cnn' else build_ann(settings, settings.augmentation)
    compile_model(model, settings)
    weights = class_weights(train_frame['class_id'])
    model_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    summary_lines = []
    model.summary(print_fn=lambda line, **kwargs: summary_lines.append(line))
    (metrics_dir / 'model_summary.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
    plot_architecture(model, settings.figures_dir / run_name / 'architecture.png')
    metadata = {'run_name': run_name, 'status': 'training', 'architecture': architecture, 'settings': settings.to_dict(), 'class_names': list(settings.class_names), 'manifest_sha256': preparation['manifest_sha256'], 'dataset_fingerprint': preparation['dataset_fingerprint'], 'groups_by_split': preparation['groups_by_split'], 'class_weights': weights, 'parameter_count': model.count_params(), 'trainable_parameter_count': int(sum((np.prod(variable.shape) for variable in model.trainable_weights))), 'layers': architecture_details(model), 'python_version': platform.python_version(), 'tensorflow_version': tf.__version__, 'sample_counts': {name: int((frame['split'] == name).sum()) for name in ('train', 'validation', 'test')}, 'monitor': 'val_loss', 'preprocessing': 'grayscale, configured interpolation to 128x128, float32 /255; augmentation only training'}
    metadata_path = model_dir / 'metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=settings.early_stopping_patience, restore_best_weights=True), tf.keras.callbacks.ModelCheckpoint(str(model_path), monitor='val_loss', save_best_only=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=settings.reduce_lr_patience, min_lr=1e-06), tf.keras.callbacks.CSVLogger(str(metrics_dir / 'epoch_log.csv')), tf.keras.callbacks.TerminateOnNaN()]
    start = time.perf_counter()
    try:
        history = model.fit(make_tf_dataset(train_frame, settings, training=True), validation_data=make_tf_dataset(validation_frame, settings), epochs=settings.epochs, class_weight=weights, callbacks=callbacks, verbose=2)
        elapsed = time.perf_counter() - start
        values = {key: [float(value) for value in data] for key, data in history.history.items()}
        if not model_path.is_file() or not values.get('val_loss') or (not all(np.isfinite(values['val_loss']))):
            raise RuntimeError('Training produced no finite validation checkpoint.')
        (metrics_dir / 'history.json').write_text(json.dumps(values, indent=2), encoding='utf-8')
        metadata.update({'status': 'trained', 'training_seconds': elapsed, 'epochs_completed': len(values['loss']), 'best_epoch': int(np.argmin(values['val_loss'])) + 1, 'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(), 'history_last_training_accuracy': values['accuracy'][-1], 'history_last_validation_accuracy': values['val_accuracy'][-1]})
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        plot_training_history(values, settings.figures_dir / run_name)
        if evaluate:
            evaluate_model(model_path, settings, run_name)
        LOGGER.info('Saved validation-selected model: %s (%.1f seconds)', model_path, elapsed)
        return metadata
    except Exception as error:
        if metadata['status'] != 'trained':
            metadata.update({'status': 'failed', 'error': f'{type(error).__name__}: {error}'})
            metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        raise


In [ ]:
"""Load provenance-checked trained models and perform consistent patch inference."""
from __future__ import annotations
import argparse
from dataclasses import dataclass
import hashlib
import io
import json
from pathlib import Path
import numpy as np
from PIL import Image, UnidentifiedImageError

@dataclass(frozen=True)
class PatchPrediction:
    class_id: int
    class_name: str
    confidence: float
    probabilities: tuple[float, ...]

def validate_probabilities(probabilities: np.ndarray, expected_samples: int) -> np.ndarray:
    probabilities = np.asarray(probabilities, dtype=np.float32)
    if probabilities.shape != (expected_samples, 7):
        raise ValueError(f'Expected ({expected_samples}, 7) probabilities; received {probabilities.shape}.')
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0) or np.any(probabilities > 1):
        raise ValueError('Model returned nonfinite or out-of-range probabilities.')
    if not np.allclose(probabilities.sum(axis=1), 1, atol=0.0001):
        raise ValueError('Model probabilities do not sum to one.')
    return probabilities

def decode_uploaded_image(data: bytes, max_pixels: int=CONFIG.max_image_pixels) -> np.ndarray:
    if not data:
        raise ValueError('The image is empty.')
    try:
        with Image.open(io.BytesIO(data)) as image:
            if image.format not in {'JPEG', 'PNG', 'BMP', 'TIFF', 'WEBP'}:
                raise ValueError('Supported formats: JPEG, PNG, BMP, TIFF, WebP.')
            if image.width * image.height > max_pixels:
                raise ValueError(f'Image exceeds the {max_pixels:,}-pixel limit.')
            image.load()
            return np.asarray(image.convert('RGB'), dtype=np.uint8)
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as error:
        raise ValueError(f'Invalid or corrupted image: {error}') from error

class Predictor:
    """A local completed training artifact; never substitutes random weights."""

    def __init__(self, model_path: str | Path, settings: Settings=CONFIG):
        self.path = Path(model_path).resolve()
        self.settings = settings
        metadata_path = self.path.parent / 'metadata.json'
        if not self.path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f'Trained model and metadata required at {self.path.parent}. Run python -m src.train.')
        self.metadata = json.loads(metadata_path.read_text())
        if self.metadata.get('status') != 'trained':
            raise ValueError('Model metadata does not describe a completed training run.')
        if tuple(self.metadata.get('class_names', [])) != settings.class_names:
            raise ValueError('Model label mapping differs from project configuration.')
        if self.metadata.get('model_sha256') != hashlib.sha256(self.path.read_bytes()).hexdigest():
            raise ValueError('Model checksum differs from its training metadata.')
        trained_settings = self.metadata.get('settings', {})
        if trained_settings.get('image_size') != settings.image_size or trained_settings.get('channels') != 1:
            raise ValueError('Model preprocessing dimensions differ from current configuration.')
        self.model = load_keras_model(self.path)
        if tuple(self.model.input_shape[1:]) != (128, 128, 1) or self.model.output_shape[-1] != 7:
            raise ValueError('Model must accept 128x128x1 and produce seven probabilities.')

    def predict_batch(self, patches: np.ndarray) -> np.ndarray:
        patches = np.asarray(patches, dtype=np.float32)
        if patches.ndim != 4 or patches.shape[1:] != (128, 128, 1) or len(patches) == 0:
            raise ValueError('Batch must have nonempty shape (N,128,128,1).')
        if not np.isfinite(patches).all() or patches.min() < 0 or patches.max() > 1:
            raise ValueError('Batch pixels must be finite and normalized to [0,1].')
        return validate_probabilities(self.model(patches, training=False).numpy(), len(patches))

    def predict_patch(self, image: np.ndarray) -> PatchPrediction:
        patch = preprocess_image(image, self.settings.image_size)
        probabilities = self.predict_batch(patch[np.newaxis])[0]
        class_id = int(probabilities.argmax())
        return PatchPrediction(class_id, CLASS_NAMES[class_id], float(probabilities[class_id]), tuple((float(value) for value in probabilities)))


In [ ]:
"""Evaluate validation-selected checkpoints using seven explicit class labels."""
from __future__ import annotations
import argparse
import json
import logging
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
LOGGER = logging.getLogger(__name__)

def classification_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=int).reshape(-1)
    if len(labels) == 0 or not np.isin(labels, range(7)).all():
        raise ValueError('Evaluation requires nonempty labels within 0..6.')
    probabilities = validate_probabilities(probabilities, len(labels))
    predicted = probabilities.argmax(axis=1)
    report = classification_report(labels, predicted, labels=list(range(7)), target_names=list(CLASS_NAMES), zero_division=0, output_dict=True)
    result = {'sample_count': len(labels), 'accuracy': float(accuracy_score(labels, predicted)), 'classification_report': report, 'confusion_matrix': confusion_matrix(labels, predicted, labels=list(range(7))).tolist()}
    for average in ('macro', 'weighted'):
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted, labels=list(range(7)), average=average, zero_division=0)
        result[average] = {'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}
    return result

def plot_training_history(history: dict, output_dir: Path) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for metric in ('accuracy', 'loss'):
        if metric not in history or f'val_{metric}' not in history:
            raise ValueError(f'History lacks training/validation {metric}.')
        figure, axis = plt.subplots(figsize=(8, 4))
        axis.plot(range(1, len(history[metric]) + 1), history[metric], color='#15836b', label='Training')
        axis.plot(range(1, len(history[f'val_{metric}']) + 1), history[f'val_{metric}'], color='#597b92', label='Validation')
        axis.set(xlabel='Epoch', ylabel=metric.capitalize(), title=f'Training and validation {metric}')
        axis.legend()
        figure.tight_layout()
        path = output_dir / f'training_{metric}.png'
        figure.savefig(path, dpi=150)
        plt.close(figure)
        outputs.append(path)
    return outputs

def plot_confusion_matrix(matrix: np.ndarray, output: Path) -> Path:
    matrix = np.asarray(matrix)
    figure, axis = plt.subplots(figsize=(10, 8))
    display = axis.imshow(matrix, cmap='Greens', vmin=0)
    figure.colorbar(display, ax=axis, label='Test patches')
    axis.set(xticks=range(7), yticks=range(7), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, xlabel='Predicted class', ylabel='True class', title='Held-out patch confusion matrix')
    plt.setp(axis.get_xticklabels(), rotation=35, ha='right')
    threshold = matrix.max() / 2 if matrix.size else 0
    for row in range(7):
        for column in range(7):
            axis.text(column, row, str(matrix[row, column]), ha='center', va='center', color='white' if matrix[row, column] > threshold else '#17231e')
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(output, dpi=150)
    plt.close(figure)
    return output

def plot_per_class(report: dict, output: Path) -> Path:
    positions = np.arange(7)
    figure, axis = plt.subplots(figsize=(11, 5))
    axis.bar(positions - 0.18, [report[name]['precision'] for name in CLASS_NAMES], 0.36, label='Precision', color='#15836b')
    axis.bar(positions + 0.18, [report[name]['recall'] for name in CLASS_NAMES], 0.36, label='Recall', color='#597b92')
    axis.set(xticks=positions, xticklabels=CLASS_NAMES, ylabel='Score', ylim=(0, 1.05), title='Per-class held-out precision and recall')
    plt.setp(axis.get_xticklabels(), rotation=25, ha='right')
    axis.legend()
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(output, dpi=150)
    plt.close(figure)
    return output

def evaluate_model(model_path: str | Path, settings: Settings=CONFIG, run_name: str | None=None) -> dict:
    predictor = Predictor(model_path, settings)
    run_name = run_name or predictor.path.parent.name
    frame, preparation = load_manifest(settings)
    if predictor.metadata['manifest_sha256'] != preparation['manifest_sha256']:
        raise ValueError('Evaluation manifest differs from the original training split.')
    results = {'run_name': run_name, 'model_sha256': predictor.metadata['model_sha256'], 'manifest_sha256': preparation['manifest_sha256'], 'class_names': list(CLASS_NAMES), 'parameter_count': predictor.metadata['parameter_count'], 'training_seconds': predictor.metadata['training_seconds'], 'splits': {}}
    test_predictions = None
    for split_name in ('train', 'validation', 'test'):
        subset = frame[frame['split'] == split_name]
        if subset.empty:
            raise ValueError(f'{split_name} split is empty.')
        probabilities = np.concatenate([predictor.predict_batch(images.numpy()) for images, _ in make_tf_dataset(subset, settings)])
        labels = subset['class_id'].to_numpy(dtype=int)
        results['splits'][split_name] = classification_metrics(labels, probabilities)
        if split_name == 'test':
            test_predictions = subset.copy()
            test_predictions['predicted_class_id'] = probabilities.argmax(axis=1)
            test_predictions['confidence'] = probabilities.max(axis=1)
            for class_id in range(7):
                test_predictions[f'probability_{class_id}'] = probabilities[:, class_id]
    metrics_dir, figures_dir = (settings.metrics_dir / run_name, settings.figures_dir / run_name)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    (metrics_dir / 'evaluation.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
    test_predictions.to_csv(metrics_dir / 'test_predictions.csv', index=False)
    test_results = results['splits']['test']
    plot_confusion_matrix(np.array(test_results['confusion_matrix']), figures_dir / 'confusion_matrix.png')
    plot_per_class(test_results['classification_report'], figures_dir / 'per_class_precision_recall.png')
    LOGGER.info('Actual test accuracy %.4f, macro F1 %.4f', test_results['accuracy'], test_results['macro']['f1'])
    return results


In [ ]:
primary_path = CONFIG.model_path
if RUN_TRAINING:
    if not primary_path.is_file():
        training_metadata = train_model(CONFIG, run_name='cnn_adam_aug')
    else:
        training_metadata = Predictor(primary_path, CONFIG).metadata
        if training_metadata['manifest_sha256'] != preparation['manifest_sha256']:
            raise RuntimeError('Existing model belongs to another prepared dataset; use the matching data or a new run.')
else:
    print('Primary training explicitly not run.')


## 25. Training History Visualization

Solid curves compare training/validation loss and accuracy by actual epoch. A widening training advantage suggests overfitting. Early stopping may produce different run lengths; do not fill missing epochs with imagined values.


In [ ]:
history_path = CONFIG.metrics_dir / 'cnn_adam_aug' / 'history.json'
if history_path.is_file():
    history = json.loads(history_path.read_text())
    plot_training_history(history, CONFIG.figures_dir / 'cnn_adam_aug')
    for metric in ('accuracy', 'loss'):
        display(Image.open(CONFIG.figures_dir / 'cnn_adam_aug' / f'training_{metric}.png'))
else:
    print('History unavailable: training has not completed.')


## 26. Model Evaluation

Accuracy, macro/weighted precision, recall and F1, all seven class supports, and prediction tables are generated from the validation-selected checkpoint. Evaluation verifies the exact original manifest checksum.


In [ ]:
evaluation = evaluate_model(CONFIG.model_path, CONFIG, 'cnn_adam_aug') if CONFIG.model_path.is_file() else None
if evaluation:
    display(evaluation['splits']['test'])
else:
    print('Evaluation not run: no completed trained model.')


## 27. Confusion Matrix

Rows are true labels and columns predicted labels. Diagonal counts are correct predictions; off-diagonal cells expose particular confusions. A class with zero support is explicitly represented.


In [ ]:
if evaluation:
    display(Image.open(CONFIG.figures_dir / 'cnn_adam_aug' / 'confusion_matrix.png'))


## 28. Classification Report

Precision is $TP/(TP+FP)$; recall is $TP/(TP+FN)$; $F_1=2PR/(P+R)$. Macro averages weight each of the seven classes equally, including absent classes with zero scores; weighted averages weight class support.


In [ ]:
if evaluation:
    display(pd.DataFrame({name: evaluation['splits']['test']['classification_report'][name] for name in CLASS_NAMES}).T)


## 29. ANN vs CNN Experiment

This full experiment implementation uses four runs for three comparisons. ANN/CNN compare no-augmentation Adam runs; optimizer and augmentation comparisons share the CNN baseline. Reused runs must match the recorded configuration and original split.


In [ ]:
"""Four actual runs support three controlled comparisons; unrun results stay unrun."""
from __future__ import annotations
import argparse
from dataclasses import replace
import json
import logging
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
EXPERIMENT_RUNS = (('ann_adam_plain', 'ann', 'adam', False), ('cnn_adam_plain', 'cnn', 'adam', False), ('cnn_sgd_plain', 'cnn', 'sgd', False), ('cnn_adam_aug', 'cnn', 'adam', True))
COMPARISONS = {'ANN vs CNN': ('ann_adam_plain', 'cnn_adam_plain'), 'Adam vs SGD': ('cnn_adam_plain', 'cnn_sgd_plain'), 'Data augmentation': ('cnn_adam_plain', 'cnn_adam_aug')}

def experiment_plan(settings: Settings=CONFIG) -> dict:
    return {'status': 'not_run', 'seed': settings.seed, 'comparisons': COMPARISONS, 'controlled_conditions': 'same source groups, batch size, epoch budget, dropout, class weights, callbacks; both optimizers use the configured learning rate', 'runs': [{'run_name': name, 'architecture': architecture, 'status': 'not_run', 'settings': replace(settings, optimizer=optimizer, augmentation=augmentation).to_dict()} for name, architecture, optimizer, augmentation in EXPERIMENT_RUNS]}

def plot_experiment_comparisons(rows: list[dict], histories: dict, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    indexed = {row['run_name']: row for row in rows}
    for comparison, names in COMPARISONS.items():
        if not all((name in indexed for name in names)):
            continue
        selected = [indexed[name] for name in names]
        figure, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].bar(names, [row['test_accuracy'] for row in selected], color=['#597b92', '#15836b'])
        axes[0].set(title='Held-out patch accuracy', ylim=(0, 1.05), ylabel='Accuracy')
        axes[1].bar(names, [row['training_seconds'] for row in selected], color=['#597b92', '#15836b'])
        axes[1].set(title='Training duration', ylabel='Seconds')
        for name, color in zip(names, ('#597b92', '#15836b')):
            axes[2].plot(range(1, len(histories[name]['val_loss']) + 1), histories[name]['val_loss'], color=color, label=name)
        axes[2].set(title='Validation loss convergence', xlabel='Epoch', ylabel='Loss')
        axes[2].legend(fontsize=8)
        for axis in axes[:2]:
            axis.tick_params(axis='x', rotation=15)
        figure.suptitle(comparison)
        figure.tight_layout()
        figure.savefig(directory / (comparison.lower().replace(' ', '_') + '.png'), dpi=150)
        plt.close(figure)
        figure, axes = plt.subplots(1, 2, figsize=(12, 4))
        for name, color in zip(names, ('#597b92', '#15836b')):
            for axis, key in zip(axes, ('loss', 'val_loss')):
                axis.plot(range(1, len(histories[name][key]) + 1), histories[name][key], color=color, label=name)
                axis.set(xlabel='Epoch', ylabel=key, title=key.replace('_', ' ').capitalize())
                axis.legend(fontsize=8)
        figure.tight_layout()
        figure.savefig(directory / (comparison.lower().replace(' ', '_') + '_losses.png'), dpi=150)
        plt.close(figure)
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.bar([row['run_name'] for row in rows], [row['test_macro_f1'] for row in rows], color='#15836b')
    axis.set(ylabel='Macro F1', ylim=(0, 1.05), title='Executed model comparison')
    axis.tick_params(axis='x', rotation=15)
    figure.tight_layout()
    figure.savefig(directory / 'model_comparison.png', dpi=150)
    plt.close(figure)

def run_experiments(settings: Settings=CONFIG, plan_only: bool=False, overwrite: bool=False) -> dict:
    plan = experiment_plan(settings)
    directory = settings.metrics_dir / 'experiments'
    directory.mkdir(parents=True, exist_ok=True)
    plan_path = directory / 'experiment_status.json'
    plan_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')
    if plan_only:
        return plan
    _, preparation = load_manifest(settings)
    plan['manifest_sha256'] = preparation['manifest_sha256']
    rows, histories = ([], {})
    for index, (name, architecture, optimizer, augmentation) in enumerate(EXPERIMENT_RUNS):
        run_settings = replace(settings, optimizer=optimizer, augmentation=augmentation)
        metadata_path = settings.models_dir / name / 'metadata.json'
        try:
            if metadata_path.is_file() and (not overwrite):
                metadata = json.loads(metadata_path.read_text())
                keys = ('seed', 'epochs', 'batch_size', 'learning_rate', 'augmentation', 'optimizer', 'dropout', 'early_stopping_patience', 'reduce_lr_patience')
                if metadata.get('status') != 'trained' or metadata.get('manifest_sha256') != preparation['manifest_sha256'] or metadata.get('architecture') != architecture or any((metadata['settings'][key] != run_settings.to_dict()[key] for key in keys)):
                    raise ValueError(f'Existing {name} differs from controlled conditions. Use --overwrite or preserve it elsewhere.')
            else:
                metadata = train_model(run_settings, architecture, name, overwrite, evaluate=False)
            result = evaluate_model(settings.models_dir / name / 'best.keras', run_settings, name)
            histories[name] = json.loads((settings.metrics_dir / name / 'history.json').read_text())
            row = {'run_name': name, 'architecture': architecture, 'optimizer': optimizer, 'augmentation': augmentation, 'train_accuracy': result['splits']['train']['accuracy'], 'validation_accuracy': result['splits']['validation']['accuracy'], 'test_accuracy': result['splits']['test']['accuracy'], 'test_macro_f1': result['splits']['test']['macro']['f1'], 'training_seconds': metadata['training_seconds'], 'parameter_count': metadata['parameter_count'], 'epochs_completed': metadata['epochs_completed'], 'best_epoch': metadata['best_epoch']}
            rows.append(row)
            plan['runs'][index]['status'] = 'completed'
            plan['runs'][index]['measured_results'] = row
            pd.DataFrame(rows).to_csv(directory / 'comparison.csv', index=False)
            plot_experiment_comparisons(rows, histories, settings.figures_dir / 'experiments')
        except Exception as error:
            plan['status'] = 'incomplete'
            plan['runs'][index]['status'] = 'failed'
            plan['runs'][index]['error'] = f'{type(error).__name__}: {error}'
            plan_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')
            raise
        plan_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')
    plan['status'] = 'completed'
    plan_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')
    return plan


In [ ]:
if RUN_FULL_EXPERIMENTS:
    experiment_results = run_experiments(CONFIG)
else:
    experiment_results = experiment_plan(CONFIG)
    print('Full experiments explicitly not run.')
comparison_path = CONFIG.metrics_dir / 'experiments' / 'comparison.csv'
if comparison_path.is_file():
    comparisons = pd.read_csv(comparison_path)
    display(comparisons[comparisons['run_name'].isin(['ann_adam_plain','cnn_adam_plain'])])


## 30. Adam vs SGD Experiment

Same architecture, initial convolution/dense weights, prepared samples, seed, batch size, epoch budget, class weights, and callback rules. Only the optimizer changes. Compare loss curves, actual duration, and accuracy at the validation-selected checkpoint.


In [ ]:
if comparison_path.is_file():
    display(comparisons[comparisons['run_name'].isin(['cnn_adam_plain','cnn_sgd_plain'])])
    display(Image.open(CONFIG.figures_dir / 'experiments' / 'adam_vs_sgd.png'))


## 31. Data Augmentation Experiment

Only augmentation changes between the two Adam CNN runs. The exact initializer seeds make network weights identical at initialization. Evaluate whether validation and held-out class-wise behavior improve; results cannot be inferred without training.


In [ ]:
if comparison_path.is_file():
    display(comparisons[comparisons['run_name'].isin(['cnn_adam_plain','cnn_adam_aug'])])
    display(Image.open(CONFIG.figures_dir / 'experiments' / 'data_augmentation.png'))


## 32. Visual Defect Localization

The complete implementation below batches overlapping windows, keeps sufficiently confident defect argmax classes, and applies class-aware NMS. $IoU=|A\cap B|/|A\cup B|$. Evaluation matches boxes one-to-one at a fixed IoU. Patch windows need not match the actual defect extent.


In [ ]:
"""Experimental sliding-window localization; windows are not object-detector boxes."""
from __future__ import annotations
import argparse
from dataclasses import asdict, dataclass
import json
import logging
from pathlib import Path
import time
from typing import TYPE_CHECKING

@dataclass(frozen=True)
class Detection:
    box: tuple[int, int, int, int]
    class_id: int
    confidence: float

@dataclass(frozen=True)
class InspectionResult:
    detections: tuple[Detection, ...]
    window_count: int
    elapsed_seconds: float

def box_iou(first, second) -> float:
    if first[2] <= first[0] or first[3] <= first[1] or second[2] <= second[0] or (second[3] <= second[1]):
        raise ValueError('IoU requires positive-area boxes.')
    overlap = max(0, min(first[2], second[2]) - max(first[0], second[0])) * max(0, min(first[3], second[3]) - max(first[1], second[1]))
    union = (first[2] - first[0]) * (first[3] - first[1]) + (second[2] - second[0]) * (second[3] - second[1]) - overlap
    return overlap / union

def non_max_suppression(detections: list[Detection], threshold: float=CONFIG.nms_iou_threshold) -> list[Detection]:
    if not 0 <= threshold <= 1:
        raise ValueError('NMS threshold must be within [0,1].')
    selected = []
    for detection in sorted(detections, key=lambda item: (-item.confidence, item.class_id, item.box)):
        if not 0 <= detection.class_id < 6 or not 0 <= detection.confidence <= 1:
            raise ValueError('Only defect IDs 0..5 and finite probabilities in [0,1] are supported.')
        if all((other.class_id != detection.class_id or box_iou(detection.box, other.box) <= threshold for other in selected)):
            selected.append(detection)
    return selected

def window_positions(length: int, window: int, stride: int) -> list[int]:
    if length <= 0 or window <= 0 or (not 1 <= stride <= window):
        raise ValueError('Positive dimensions and stride within window size are required.')
    if length <= window:
        return [0]
    positions = list(range(0, length - window + 1, stride))
    if positions[-1] != length - window:
        positions.append(length - window)
    return positions

def inspect_image(image, predictor: Predictor, settings: Settings=CONFIG) -> InspectionResult:
    """Stream bounded batches; include image edges, threshold defects, then class-aware NMS."""
    import numpy as np
    settings.validate()
    image = np.asarray(image)
    if image.ndim not in (2, 3) or image.size == 0 or (not np.isfinite(image).all()):
        raise ValueError('Inspection requires a finite nonempty image.')
    height, width = image.shape[:2]
    if height * width > settings.max_image_pixels:
        raise ValueError('Image exceeds the configured pixel limit.')
    xs = window_positions(width, settings.window_size, settings.stride)
    ys = window_positions(height, settings.window_size, settings.stride)
    window_count = len(xs) * len(ys)
    if window_count > 4096:
        raise ValueError(f'{window_count} windows required; increase window size/stride to stay below 4096.')
    start = time.perf_counter()
    candidates, batch, coordinates = ([], [], [])

    def flush():
        if not batch:
            return
        probabilities = predictor.predict_batch(np.stack(batch))
        for box, distribution in zip(coordinates, probabilities):
            class_id = int(distribution.argmax())
            confidence = float(distribution[class_id])
            if class_id != 6 and confidence >= settings.confidence_threshold:
                candidates.append(Detection(box, class_id, confidence))
        batch.clear()
        coordinates.clear()
    for y in ys:
        for x in xs:
            box = (x, y, min(x + settings.window_size, width), min(y + settings.window_size, height))
            batch.append(preprocess_image(image[y:box[3], x:box[2]], settings.image_size))
            coordinates.append(box)
            if len(batch) == settings.batch_size:
                flush()
    flush()
    return InspectionResult(tuple(non_max_suppression(candidates, settings.nms_iou_threshold)), window_count, time.perf_counter() - start)

def draw_detections(image, detections):
    import numpy as np
    from PIL import Image, ImageDraw
    canvas = Image.fromarray(np.asarray(image).astype(np.uint8)).convert('RGB')
    drawing = ImageDraw.Draw(canvas)
    for detection in detections:
        x1, y1, x2, y2 = detection.box
        color = '#15836b'
        drawing.rectangle((x1, y1, x2 - 1, y2 - 1), outline=color, width=2)
        label = f'{CLASS_NAMES[detection.class_id]} {detection.confidence:.1%}'
        bounds = drawing.textbbox((0, 0), label)
        text_width, text_height = (bounds[2] - bounds[0], bounds[3] - bounds[1])
        tx = max(0, min(x1, canvas.width - text_width - 4))
        ty = max(0, y1 - text_height - 6)
        drawing.rectangle((tx, ty, min(canvas.width - 1, tx + text_width + 4), ty + text_height + 5), fill=color)
        drawing.text((tx + 2, ty), label, fill='white')
    return canvas

def match_detections(predictions: list[Detection], annotations, iou_threshold: float=CONFIG.localization_iou_threshold) -> dict:
    """Greedy class-aware one-to-one matching, without invented background true negatives."""
    if not 0 < iou_threshold <= 1:
        raise ValueError('Matching threshold must be within (0,1].')
    used, matched_ious = (set(), [])
    per_class = {str(class_id): {'tp': 0, 'fp': 0, 'fn': 0} for class_id in range(6)}
    for prediction in sorted(predictions, key=lambda item: -item.confidence):
        if prediction.class_id not in range(6):
            raise ValueError('Localization predictions must be defect classes 0..5.')
        candidates = [(index, box_iou(prediction.box, annotation.box)) for index, annotation in enumerate(annotations) if index not in used and annotation.class_id == prediction.class_id]
        best = max(candidates, key=lambda item: item[1], default=None)
        if best is not None and best[1] >= iou_threshold:
            used.add(best[0])
            matched_ious.append(best[1])
            per_class[str(prediction.class_id)]['tp'] += 1
        else:
            per_class[str(prediction.class_id)]['fp'] += 1
    for index, annotation in enumerate(annotations):
        if index not in used:
            per_class[str(annotation.class_id)]['fn'] += 1
    tp, fp, fn = (sum((value[key] for value in per_class.values())) for key in ('tp', 'fp', 'fn'))
    return {'tp': tp, 'fp': fp, 'fn': fn, 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'matched_ious': matched_ious, 'mean_matched_iou': sum(matched_ious) / len(matched_ious) if matched_ious else 0.0, 'per_class': per_class}

def evaluate_localization(predictor: Predictor, settings: Settings=CONFIG, split: str='test') -> dict:
    if split not in {'validation', 'test'}:
        raise ValueError('Localization evaluation uses validation or test images.')
    frame, preparation = load_manifest(settings)
    if predictor.metadata['manifest_sha256'] != preparation['manifest_sha256']:
        raise ValueError("Localization must use the model's original source split.")
    images = frame[frame['split'] == split].drop_duplicates('image_id')
    if images.empty:
        raise ValueError(f'No source images in {split}.')
    matches, records = ([], [])
    for row in images.itertuples():
        image = read_image(row.source_image, settings.max_image_pixels)
        annotations = parse_annotations(row.annotation_path, preparation['settings']['annotation_origin'])
        for annotation in annotations:
            validate_box(annotation.box, image.shape[1], image.shape[0])
        inspection = inspect_image(image, predictor, settings)
        match = match_detections(list(inspection.detections), annotations, settings.localization_iou_threshold)
        matches.append(match)
        records.append({'image_id': row.image_id, 'window_count': inspection.window_count, 'elapsed_seconds': inspection.elapsed_seconds, 'detections': [asdict(item) for item in inspection.detections], **match})
    tp, fp, fn = (sum((match[key] for match in matches)) for key in ('tp', 'fp', 'fn'))
    ious = [value for match in matches for value in match['matched_ious']]
    per_class = {}
    for class_id in range(6):
        values = {key: sum((match['per_class'][str(class_id)][key] for match in matches)) for key in ('tp', 'fp', 'fn')}
        values['precision'] = values['tp'] / (values['tp'] + values['fp']) if values['tp'] + values['fp'] else 0.0
        values['recall'] = values['tp'] / (values['tp'] + values['fn']) if values['tp'] + values['fn'] else 0.0
        per_class[CLASS_NAMES[class_id]] = values
    return {'scope': 'experimental window localization; not detector mAP', 'split': split, 'image_count': len(records), 'model_sha256': predictor.metadata['model_sha256'], 'manifest_sha256': preparation['manifest_sha256'], 'settings': settings.to_dict(), 'tp': tp, 'fp': fp, 'fn': fn, 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'mean_matched_iou': sum(ious) / len(ious) if ious else 0.0, 'per_class': per_class, 'images': records}


In [ ]:
if CONFIG.model_path.is_file():
    predictor = Predictor(CONFIG.model_path, CONFIG)
    localization_metrics = evaluate_localization(predictor, CONFIG, split='test')
    display({key: localization_metrics[key] for key in ('image_count','precision','recall','mean_matched_iou')})
    CONFIG.metrics_dir.joinpath('cnn_adam_aug','localization_test.json').write_text(json.dumps(localization_metrics, indent=2))


## 33. Sample Predictions

Sample predictions come from actual held-out prepared patches. We display true and predicted classes only after a completed model is loaded; random-model examples are not presented as results.


In [ ]:
if CONFIG.model_path.is_file():
    predictor = Predictor(CONFIG.model_path, CONFIG)
    sample_rows = frame[frame['split']=='test'].head(6)
    figure, axes = plt.subplots(2,3,figsize=(12,7))
    for axis, row in zip(axes.flat, sample_rows.itertuples()):
        patch = read_image(CONFIG.processed_dir / row.patch_path)
        prediction = predictor.predict_patch(patch)
        axis.imshow(patch, cmap='gray'); axis.axis('off')
        axis.set_title(f'True: {CLASS_NAMES[row.class_id]}\nPredicted: {prediction.class_name} {prediction.confidence:.1%}', fontsize=9)
    plt.tight_layout(); plt.show()


## 34. Model Saving

ModelCheckpoint writes the best `.keras` file during actual training. Metadata records the checkpoint SHA256, dataset manifest checksum, class mapping and preprocessing. We do not save the untrained teaching model as a trained project artifact.


In [ ]:
if CONFIG.model_path.is_file():
    print('Validation-selected artifact:', CONFIG.model_path)
    print('SHA256:', hashlib.sha256(CONFIG.model_path.read_bytes()).hexdigest())
else:
    print('No trained model has been saved.')


## 35. Model Loading

`load_keras_model` handles registered sparse metrics. Production `Predictor` additionally requires completed-training metadata, matching class order, dimensions and checksum.


In [ ]:
if CONFIG.model_path.is_file():
    loaded_predictor = Predictor(CONFIG.model_path, CONFIG)
    print('Loaded actual trained run:', loaded_predictor.metadata['run_name'])


## 36. Inference on New PCB Images

Set `NEW_IMAGE_PATH` to an actual image. Uploaded decoding rejects unsupported/corrupt files and oversized images. Use patch prediction for one crop; use sliding-window inspection for a full board.


In [ ]:
NEW_IMAGE_PATH = None
if NEW_IMAGE_PATH is not None:
    if not CONFIG.model_path.is_file():
        raise FileNotFoundError('Train the CNN before new-image inference.')
    new_image = decode_uploaded_image(Path(NEW_IMAGE_PATH).read_bytes())
    new_inspection = inspect_image(new_image, Predictor(CONFIG.model_path, CONFIG), CONFIG)
    display(draw_detections(new_image, new_inspection.detections))
else:
    print('No new image selected; upload one through app.py or set NEW_IMAGE_PATH.')


## 37. Results and Observations

Only measured artifacts are shown. In this development environment dataset download, scientific dependencies and TensorFlow execution were blocked; no accuracy, confusion matrix, or trained checkpoint was fabricated. After execution, use support counts and per-class F1 as well as overall accuracy.


In [ ]:
if comparison_path.is_file():
    display(pd.read_csv(comparison_path))
else:
    print('Experimental results not available: experiments have not completed.')


## 38. Limitations

Annotation completeness governs normal labels. Board-group splitting can expose distribution shift. Tight crops lose trace context; whole-board windows can miss tiny defects. A single seed does not estimate variance. Scores are uncalibrated and optical inspection does not establish electrical continuity.


## 39. Future Scope

Evaluate reference/test difference channels as a separately documented model, collect more board designs, repeat seeds, calibrate on validation data, and study dedicated detection models separately. Avoid presenting this patch classifier as an industrial detector.


## 40. Final Conclusion

The project explicitly connects neurons and feature extraction to cross-entropy, backpropagation, Adam/SGD and regularization. Completion requires real dataset inspection, all tests, actual training, evaluation and app validation. The code here is a full inline implementation; output cells remain unexecuted until those resources are available.
